# 1.2 Tools — Giving Agents the Ability to Act

## What you will learn in this notebook

- What a **tool** is and why agents need them
- Three ways to **define a tool** using the `@tool` decorator
- How to **call a tool directly** (without an agent) to test it
- How to **attach tools to an agent** so the model can decide when to use them
- How to **inspect the agent's reasoning** — what tool it called, with what arguments, and what came back

---

### Why do agents need tools?

A plain LLM is a **frozen snapshot of knowledge** — it cannot:
- Look up today's stock price
- Run a precise calculation (LLMs are notoriously bad at maths)
- Query your database
- Send an email
- Call an external API

**Tools** bridge this gap. A tool is just a **Python function** that the agent is allowed to call. The model decides *when* to call it and *what arguments* to pass — you just define *what it does*.

```
Without tools:  "What is √467?"  →  LLM guesses  →  "approximately 21.6"  (may be wrong)
With tools:     "What is √467?"  →  LLM calls square_root(467)  →  21.587...  (exact)
```

### How does the model know which tool to call?

LangChain sends the model a **description of every available tool** alongside the user's message. The model reads those descriptions and decides:
1. Do I need a tool to answer this?
2. If yes, which tool fits best?
3. What arguments should I pass?

This means **the tool's name and docstring are part of your prompt**. Clear, descriptive names and docstrings = the model makes better decisions.

---

### The tool call lifecycle

```
User: "What is √467?"
         │
         ▼
    ┌─────────┐   "I need to call square_root(x=467)"
    │   LLM   │ ─────────────────────────────────────▶ ┌──────────────────┐
    │ (brain) │                                         │  square_root(467) │
    │         │ ◀─────────────────────────────────────  │  returns 21.587  │
    └─────────┘   ToolMessage(content="21.587...")       └──────────────────┘
         │
         ▼
    "The square root of 467 is approximately 21.59"
```

The message history after this call looks like:
```
[0] HumanMessage   → "What is √467?"
[1] AIMessage      → (empty content, but has tool_calls=[{name: square_root, args: {x: 467}}])
[2] ToolMessage    → "21.587..."
[3] AIMessage      → "The square root of 467 is approximately 21.59"
```

---
## Section 1 — Defining Tools

LangChain provides the `@tool` decorator to turn any ordinary Python function into a tool an agent can use.

When you apply `@tool`, LangChain automatically:
1. Extracts the **function name** → used as the tool's identifier
2. Extracts the **docstring** → sent to the model as the tool description
3. Reads the **type hints** → builds a JSON Schema so the model knows what arguments to pass
4. Wraps the function with an `.invoke()` method so LangGraph can call it uniformly

There are three syntax variants — all produce equivalent tools.

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Reads OPENAI_API_KEY (and others) from the .env file.
# Must run before any LangChain code that hits an external API.

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# ============================================================
# CELL 2: Tool Definition — Style A (Simplest / Most Common)
# ============================================================
# @tool with no arguments — the decorator reads everything from
# the function itself:
#
#   Function name  → tool name:        "square_root"
#   Docstring      → tool description: "Calculate the square root of a number"
#   Type hints     → argument schema:  {x: float}
#
# The docstring is CRITICAL. The model reads it to decide whether
# to call this tool. Bad docstring = model won't know when to use it.
#   ✗ Bad:  "does math"
#   ✓ Good: "Calculate the square root of a number. Input x must be >= 0."
#
# Type hints (x: float) are also important — they tell LangChain
# to build a JSON Schema that the model uses when constructing
# the tool call arguments. Always annotate your tool parameters.

from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
# ============================================================
# CELL 3: Tool Definition — Style B (Custom Tool Name)
# ============================================================
# @tool("custom_name") lets you override the tool name while
# keeping the docstring as the description.
#
# When would you use this?
#   - When the function has a generic name (tool1, helper, fn)
#     but you want the tool exposed with a descriptive name
#   - When refactoring: you can rename the Python function
#     without changing the tool name the agent was trained on
#
# Here the Python function is named 'tool1' (generic) but the
# agent sees it as 'square_root' (descriptive).
# The docstring still provides the description.

@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
# ============================================================
# CELL 4: Tool Definition — Style C (Custom Name + Description)
# ============================================================
# @tool("name", description="...") lets you override both the
# tool name AND the description independently of the function
# name and docstring.
#
# When would you use this?
#   - When the docstring is written for human developers
#     (e.g. includes implementation notes) but you want a
#     cleaner, model-facing description
#   - When you want the description to be richer without
#     cluttering the docstring
#   - In tool libraries where the function name follows a
#     different convention than what the model should see
#
# This is the most explicit and readable style — recommended
# when building production tools that other developers will maintain.
#
# Note: no docstring here — the description kwarg replaces it.
from langchain.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def square_root(x: float) -> float:
    return x ** 0.5

In [5]:
# ============================================================
# CELL 5: Test the Tool Directly (Without an Agent)
# ============================================================
# ALWAYS test your tools in isolation before attaching them to
# an agent. This way if something breaks, you know whether the
# bug is in the tool itself or in the agent's decision-making.
#
# tool1.invoke() calls the underlying Python function directly.
# It bypasses the LLM entirely — no API call, no cost.
#
# We pass a dict matching the tool's parameter schema:
#   {"x": 467}  →  calls tool1(x=467)  →  returns 21.587...
#
# If this cell works correctly, you know the tool logic is sound.
# If it fails here, fix the tool before involving the agent.

square_root.invoke({"x": 467})

21.61018278497431

---
## Section 2 — Attaching Tools to an Agent

Once a tool is defined and tested, we give it to an agent via the `tools=` parameter.

When the agent receives a question, it now has two choices:
1. **Answer directly** from its own knowledge (if it can)
2. **Call a tool** (if the question requires computation, data lookup, etc.)

The model makes this decision automatically based on the question and the tool descriptions. You do not need to tell it when to use the tool — it figures that out.

### What happens in the message history?

When a tool is called, LangGraph adds messages automatically:
- `messages[0]` → `HumanMessage` — the user's question
- `messages[1]` → `AIMessage` — the model's decision to call the tool (`.content` is empty, `.tool_calls` has the call details)
- `messages[2]` → `ToolMessage` — the result returned by the tool function
- `messages[3]` → `AIMessage` — the model's final answer (wraps the tool result in natural language)

That's why we use `[-1]` to get the final answer — it's always the last message regardless of how many tool calls happened.

In [6]:
# ============================================================
# CELL 6: Create an Agent with a Tool
# ============================================================
# tools=[tool1] gives the agent access to our square_root tool.
#
# You can pass multiple tools: tools=[tool1, tool2, tool3]
# The agent will choose the appropriate one based on the question.
#
# The system_prompt here does two things:
#   1. Gives the agent a persona ("arithmetic wizard")
#   2. Tells it what capabilities it has ("Use your tools...")
#      This is a nudge — the model would figure it out from the
#      tool descriptions, but an explicit reminder helps.
#
# 💡 Tip: when writing system prompts for tool-enabled agents,
#    mention the tools by their capability, not by name:
#      ✓ "Use your tools to calculate exact values"
#      ✗ "Use the square_root tool"  (too prescriptive — limits flexibility)

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[square_root],   # List of tool objects — agent can call any of these
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [8]:
# ============================================================
# CELL 7: Ask the Agent a Question That Requires the Tool
# ============================================================
# The model receives: user question + tool descriptions.
# It recognises "square root of 467" matches our tool's description
# and decides to call it instead of guessing.
#
# We use response['messages'][-1].content to get the final answer.
#
# Why [-1] instead of [1]?
#   When a tool is called, there are 4 messages in the history
#   (Human → AIMessage(tool_call) → ToolMessage → AIMessage(final)).
#   Using [1] would give us the intermediate AIMessage that just
#   contains a tool call request — its .content is empty!
#   Using [-1] always gives the LAST message = the final answer.
#   This works for both tool calls (4 messages) and direct answers
#   (2 messages) — making [-1] the safest general pattern.

from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square roots of 234?")

response = agent.invoke(
    {"messages": [question]}
)

print(response)

# [-1] = last message = the agent's final natural-language answer
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='What is the square roots of 234?', additional_kwargs={}, response_metadata={}, id='aaa0dc7e-597e-4383-b475-8594681f4e59'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 599, 'prompt_tokens': 158, 'total_tokens': 757, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DwfjSQaj4LpeizkHugRbOwvThzxwX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1bad-f675-7331-93ee-4e37a83c5f35-0', tool_calls=[{'name': 'square_root', 'args': {'x': 234}, 'id': 'call_JqGdmPie4GKADneBG2fCYtfk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 59

In [9]:
# ============================================================
# CELL 7: Ask the Agent a Question That Requires the Tool
# ============================================================
# The model receives: user question + tool descriptions.
# It recognises "square root of 467" matches our tool's description
# and decides to call it instead of guessing.
#
# We use response['messages'][-1].content to get the final answer.
#
# Why [-1] instead of [1]?
#   When a tool is called, there are 4 messages in the history
#   (Human → AIMessage(tool_call) → ToolMessage → AIMessage(final)).
#   Using [1] would give us the intermediate AIMessage that just
#   contains a tool call request — its .content is empty!
#   Using [-1] always gives the LAST message = the final answer.
#   This works for both tool calls (4 messages) and direct answers
#   (2 messages) — making [-1] the safest general pattern.

from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square roots of respective numbers which are pipe separated 467|667|234?")



response = agent.invoke(
    {"messages": [question]}
)

print(response)

# [-1] = last message = the agent's final natural-language answer
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='What is the square roots of respective numbers which are pipe separated 467|667|234?', additional_kwargs={}, response_metadata={}, id='cdd37328-f3d4-4d6d-9ffc-a0e48f383d24'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2499, 'prompt_tokens': 168, 'total_tokens': 2667, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2432, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DwfvlokMLQKcYGBCsQjOWLqJZxsH0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1bb9-9d86-7943-8751-10f885cdb67b-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_XAKpB5b2pIz2uWcDLCV3vZPr', 'type': 'tool_call'}, {'name': 'square_root', '

In [10]:
# ============================================================
# CELL 8: Inspect the Full Message History
# ============================================================
# This is one of the most valuable debugging techniques for agents.
# Printing the full message list lets you see exactly what happened
# inside the agent loop:
#
#   messages[0]  HumanMessage
#       content: "What is the square root of 467?"
#
#   messages[1]  AIMessage
#       content: ""  (empty — the model is making a tool call, not replying yet)
#       tool_calls: [{name: "square_root", args: {x: 467}, id: "..."}]
#
#   messages[2]  ToolMessage
#       content: "21.587..."  (the actual return value of our Python function)
#       tool_call_id: "..."   (links back to the AIMessage that requested it)
#
#   messages[3]  AIMessage
#       content: "The square root of 467 is approximately 21.59."
#       (the model wraps the tool result in a natural language reply)
#
# Use this pattern whenever your agent gives unexpected answers:
#   - Did it call the tool at all?
#   - Did it pass the right arguments?
#   - What did the tool actually return?

from pprint import pprint
for messasge in response['messages']:
    print("****************************")
    pprint(messasge)

****************************
HumanMessage(content='What is the square roots of respective numbers which are pipe separated 467|667|234?', additional_kwargs={}, response_metadata={}, id='cdd37328-f3d4-4d6d-9ffc-a0e48f383d24')
****************************
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2499, 'prompt_tokens': 168, 'total_tokens': 2667, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2432, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DwfvlokMLQKcYGBCsQjOWLqJZxsH0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1bb9-9d86-7943-8751-10f885cdb67b-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_XAKpB5b2pIz2uWcDLCV3vZPr', 'typ

In [ ]:
# ============================================================
# CELL 9: Inspect the Tool Call Decision
# ============================================================
# messages[1] is the AIMessage where the model decided to use a tool.
# Its .tool_calls attribute is a list of tool call requests.
#
# Each tool call dict contains:
#   'name'  → which tool the model chose ("square_root")
#   'args'  → the arguments it decided to pass ({"x": 467.0})
#   'id'    → a unique ID that links this call to the ToolMessage response
#   'type'  → always "tool_call"
#
# This is where you can verify that the model interpreted the user's
# intent correctly:
#   - Correct tool selected? ✓
#   - Correct argument extracted from the question? ✓
#   - Argument type correct (float, not string)? ✓
#
# If any of these are wrong, refine your tool description or
# system prompt to guide the model better.

print(response["messages"][1].tool_calls)

---
## Summary — Tools in a Nutshell

| Concept | Key takeaway |
|---|---|
| **Why tools?** | LLMs can't do real-time lookups, precise maths, or side effects — tools fill that gap |
| **`@tool` decorator** | Turns any Python function into a LangChain-compatible tool |
| **Docstring = description** | The model reads the docstring to decide when to call the tool — write it for the model, not just humans |
| **Type hints** | Required — LangChain uses them to build the argument schema the model fills in |
| **Test with `.invoke()`** | Always test tools in isolation before attaching to an agent |
| **`tools=[...]`** | Pass a list of tool objects to `create_agent` — agent can use any of them |
| **`messages[-1]`** | Always use the last message for the final answer — safe for both direct replies and tool calls |
| **`messages[1].tool_calls`** | Shows exactly which tool was called, with what args — essential for debugging |

### The 4-message pattern (memorise this)

```
[0] HumanMessage      → user's question
[1] AIMessage         → model's tool call decision  (.content is empty, .tool_calls has the call)
[2] ToolMessage       → tool's return value
[3] AIMessage         → model's final natural-language answer  ← response['messages'][-1]
```

### Golden rules for tool design

1. **One tool, one job** — tools should do one thing well, not many things loosely
2. **Name and describe for the model** — the model reads these, not just your teammates
3. **Always annotate types** — `x: float`, `query: str`, etc. — no bare parameters
4. **Test in isolation first** — run `.invoke({...})` before plugging into an agent
5. **Return simple types** — strings, numbers, dicts. Complex objects confuse the model when it reads the ToolMessage

### Next steps
- **1.3** — Memory: persisting conversation context across sessions
- **1.4** — Multi-tool agents: giving the agent a toolkit and letting it choose